# M2 — InternVideo2 features + QD-DETR head on Charades-STA

Pipeline:
1. Mount Drive + sync the `tl-jockey` repo onto Colab
2. Install training deps (transformers, decord, einops, timm)
3. Extract InternVideo2 features for Charades videos → `features/iv2_charades/`
4. Precompute IV2 text-tower query embeddings (aligned with the visual features)
5. Train `QDDETRHead`
6. Inspect results, compare with the existing `charades_incremental` run

All heavy steps are resumable: `--skip-existing` for features, `--resume` for training. Re-run any cell as needed.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the repo onto Colab

Default: rsync from `MyDrive/tl-jockey` (keep your working copy there).
Fallback: uncomment the `git clone` line and set `REPO_URL`.

We exclude `features/`, `runs/`, `data/` from the sync — those live in `MyDrive/data/` and we don't want to bloat Colab's local disk.

In [ ]:
import os, sys, subprocess

REPO_NAME  = 'tl-jockey'
COLAB_REPO = f'/content/{REPO_NAME}'

# === Pick ONE source. Easiest: option (A). ===
# (A) Git clone -- set REPO_URL to your fork. Push your local repo to GitHub first:
#     cd ~/MyProj/tl-jockey && git add . && git commit -m 'm2' && git push
REPO_URL = ''                # e.g. 'https://github.com/HaiVD16/tl-jockey'
BRANCH   = 'main'
# (B) rsync from Drive -- set this if you uploaded the repo via the Drive web UI.
DRIVE_REPO = f'/content/drive/MyDrive/{REPO_NAME}'

if REPO_URL:
    if os.path.isdir(COLAB_REPO):
        print(f'pulling latest in {COLAB_REPO}')
        subprocess.run(['git', '-C', COLAB_REPO, 'fetch', '--depth=1', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', COLAB_REPO, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
    else:
        print(f'git clone {REPO_URL} (branch {BRANCH}) -> {COLAB_REPO}')
        subprocess.run(
            ['git', 'clone', '--depth=1', '--branch', BRANCH, REPO_URL, COLAB_REPO],
            check=True,
        )
elif os.path.isdir(DRIVE_REPO):
    print(f'rsync {DRIVE_REPO} -> {COLAB_REPO}')
    subprocess.run([
        'rsync', '-a', '--delete',
        '--exclude', '.git', '--exclude', '__pycache__', '--exclude', '.venv',
        '--exclude', 'features', '--exclude', 'runs', '--exclude', 'data',
        f'{DRIVE_REPO}/', f'{COLAB_REPO}/',
    ], check=True)
else:
    raise SystemExit(
        'No repo source configured. Pick ONE:\n'
        '  (A) Set REPO_URL above to your fork. Push your local repo first:\n'
        "      cd ~/MyProj/tl-jockey && git add . && git commit -m 'm2' && git push\n"
        f'  (B) Upload the repo to {DRIVE_REPO} (Drive web UI: drag the tl-jockey folder into MyDrive).'
    )

os.chdir(COLAB_REPO)
if COLAB_REPO not in sys.path:
    sys.path.insert(0, COLAB_REPO)
print('cwd:', os.getcwd())
print('training pkg present:',
      os.path.isdir(os.path.join(COLAB_REPO, 'jockey/open_source/training')))


## 3. Install training deps

Just what M2 needs — no langgraph/langchain (those are agent-side).

In [ ]:
!pip install --quiet --upgrade \
    'transformers>=4.40' 'decord>=0.6.0' 'einops>=0.7.0' 'timm>=0.9.16' \
    'scenedetect[opencv]>=0.6.4'
print('deps installed')

## 4. Paths

Mirrors your existing Drive layout. The new IV2 paths live alongside the existing CLIP ones — nothing is overwritten.

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/data'

# Your existing layout (untouched)
ANN_DIR     = f'{DRIVE_ROOT}'
FULL_DIR    = f'{DRIVE_ROOT}/features/charades'
QUERY_CACHE = f'{FULL_DIR}/query_emb.npz'
RUN_DIR     = f'{DRIVE_ROOT}/runs/charades_incremental'

# New IV2 layout
IV2_FEAT_DIR    = f'{DRIVE_ROOT}/features/iv2_charades'
IV2_QUERY_CACHE = f'{IV2_FEAT_DIR}/query_emb_iv2.npz'
IV2_RUN_DIR     = f'{DRIVE_ROOT}/runs/qd_detr_iv2'

# Charades raw MP4s. If you don't have these on Drive, you can skip Step 5
# (extract features off-Colab, upload .npz files to IV2_FEAT_DIR, then start at Step 6).
VIDEOS_DIR = f'{DRIVE_ROOT}/charades/videos'

TRAIN_ANN = f'{ANN_DIR}/charades_sta_train.txt'
TEST_ANN  = f'{ANN_DIR}/charades_sta_test.txt'

os.makedirs(IV2_FEAT_DIR, exist_ok=True)
os.makedirs(IV2_RUN_DIR, exist_ok=True)

for p in [TRAIN_ANN, TEST_ANN, VIDEOS_DIR, IV2_FEAT_DIR, FULL_DIR]:
    print(('OK     ' if os.path.exists(p) else 'MISSING'), p)

## 5. Environment check

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram total (GiB):', f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}')

n_train = sum(1 for _ in open(TRAIN_ANN)) if os.path.isfile(TRAIN_ANN) else 0
n_test  = sum(1 for _ in open(TEST_ANN))  if os.path.isfile(TEST_ANN)  else 0
print(f'annotations: train={n_train}  test={n_test}')

if os.path.isdir(FULL_DIR):
    n_clip = len([f for f in os.listdir(FULL_DIR) if f.endswith('.npz') and f != 'query_emb.npz'])
    print(f'existing CLIP feature files in {FULL_DIR}: {n_clip}')
if os.path.isdir(IV2_FEAT_DIR):
    n_iv2 = len([f for f in os.listdir(IV2_FEAT_DIR) if f.endswith('.npz') and not f.startswith('query')])
    print(f'existing IV2 feature files in {IV2_FEAT_DIR}: {n_iv2}')

## 6. Smoke test — extract IV2 features for 2 videos

Run this before the full batch extraction. Catches model-load issues, dim mismatches, OOM at clip-batch size, etc.

**T4 VRAM tip**: if you OOM during encoding, drop `--encode-batch-size` from 8 to 4.

In [ ]:
import glob
sample = sorted(glob.glob(f'{VIDEOS_DIR}/**/*.mp4', recursive=True))[:2]
if not sample:
    print(f'No MP4s under {VIDEOS_DIR}.')
    print('Set VIDEOS_DIR in cell 4, or skip Steps 6-7 and upload pre-extracted')
    print('IV2 .npz files to IV2_FEAT_DIR before running Step 8.')
else:
    print(f'smoke-testing on {len(sample)} videos')
    for v in sample:
        vid = os.path.splitext(os.path.basename(v))[0]
        out = f'{IV2_FEAT_DIR}/{vid}.npz'
        cmd = (
            f'python -m jockey.open_source.training.iv2_feature_extractor '
            f'--video "{v}" --out "{out}" '
            f'--window-sec 2.0 --frames-per-clip 4 --dtype fp16 '
            f'--encode-batch-size 8'
        )
        !{cmd}

    # Peek at one output
    import numpy as np
    one = f'{IV2_FEAT_DIR}/{os.path.splitext(os.path.basename(sample[0]))[0]}.npz'
    if os.path.isfile(one):
        d = np.load(one, allow_pickle=True)
        print('\nOutput schema check:')
        print(f'  visual_features: {d["visual_features"].shape} dtype={d["visual_features"].dtype}')
        print(f'  shot_boundaries: {d["shot_boundaries"].shape}')
        print(f'  duration       : {float(d["duration"]):.1f}s')
        v = d['visual_features']
        norms = np.linalg.norm(v, axis=1)
        print(f'  L2 norms       : mean={norms.mean():.3f}  std={norms.std():.3f}')
        print(f'                   (expect ~1.0 if encoder L2-normalized)')

## 7. Full batch extraction

On a T4, expect roughly **10–15 s per ~30 s Charades video** at 4 frames/clip, fp16. Charades has ~5500 train videos + ~1300 test → ~20-25 hours total. **Use `--skip-existing` so you can chain multiple Colab sessions** (free tier kicks you off after ~12 h).

In [ ]:
cmd = (
    f'python -m jockey.open_source.training.iv2_batch_extract '
    f'--videos-dir "{VIDEOS_DIR}" '
    f'--out-dir "{IV2_FEAT_DIR}" '
    f'--window-sec 2.0 '
    f'--frames-per-clip 4 '
    f'--dtype fp16 '
    f'--encode-batch-size 8 '
    f'--skip-existing'
)
!{cmd}

## 8. Precompute IV2 query embeddings

Embeds Charades-STA queries with the InternVideo2 **text tower** — same space as the visual features. `--features-dir-filter` keeps only queries whose video_id has features (lets you train on a subset before the full extraction is done).

In [ ]:
cmd = (
    f'python -m jockey.open_source.training.precompute_queries '
    f'--annotations "{TRAIN_ANN}" "{TEST_ANN}" '
    f'--out "{IV2_QUERY_CACHE}" '
    f'--encoder iv2 '
    f'--features-dir-filter "{IV2_FEAT_DIR}"'
)
!{cmd}

## 9. Train QDDETRHead

Defaults: 30 epochs, batch 32, hidden 256, 2 self-attn layers, mixed precision. ~3-5M trainable params — fits T4 with plenty of headroom.

To resume after a Colab disconnect: re-run with `--resume {IV2_RUN_DIR}/last.pt` appended.

In [ ]:
cmd = (
    f'python -m jockey.open_source.training.qd_detr_train '
    f'--features-dir "{IV2_FEAT_DIR}" '
    f'--train-ann "{TRAIN_ANN}" '
    f'--test-ann "{TEST_ANN}" '
    f'--query-cache "{IV2_QUERY_CACHE}" '
    f'--out-dir "{IV2_RUN_DIR}" '
    f'--epochs 30 '
    f'--batch-size 32 '
    f'--hidden-dim 256 '
    f'--num-self-layers 2 '
    f'--lr 2e-4 '
    f'--mixed-precision'
)
!{cmd}

## 10. Inspect & compare with the existing run

The number to write in the thesis is the final test-set R@1@IoU=0.5. We also pull the existing ViCLIP+GroundingHead run for an apples-to-apples comparison.

In [ ]:
import json

def show_metrics(label, path):
    if os.path.isfile(path):
        print(f'=== {label} ({path}) ===')
        print(json.dumps(json.load(open(path)), indent=2))
        print()
    else:
        print(f'(no {label} at {path})')

show_metrics('IV2 + QDDETRHead',         f'{IV2_RUN_DIR}/test_metrics.json')
show_metrics('ViCLIP + GroundingHead',   f'{RUN_DIR}/test_metrics.json')

# Val curve tail for the IV2 run
val_log = f'{IV2_RUN_DIR}/val_log.csv'
if os.path.isfile(val_log):
    lines = open(val_log).read().splitlines()
    print('val_log.csv (header + last 10 epochs):')
    print(lines[0])
    for ln in lines[-10:]:
        print(ln)

### Next steps for the thesis ablation

Once this run completes, the encoder-swap ablation (Option A from the research synthesis) is one extra training run away:

- **Variant 1**: existing ViCLIP + existing GroundingHead → already in `runs/charades_incremental/`
- **Variant 2**: ViCLIP features + `QDDETRHead` → re-run cell 9 with `--features-dir {FULL_DIR} --query-cache {QUERY_CACHE}` (isolates the head architecture)
- **Variant 3**: InternVideo2 features + `QDDETRHead` → this notebook (changes both encoder and head)
- **Variant 4** (optional): InternVideo2 features + original `GroundingHead` → use `train.py` with the new features dir + IV2 query cache (isolates the encoder)

Four rows of R@1@0.5 / R@1@0.7 = one publishable ablation table.